# Qwen2.5-1.5B + LoRA: `[TRACK]` tracker

Runs [`training/qwen_track.py`](qwen_track.py): trains the tracker's LoRA adapter on the MultiWOZ +
synthetic `[TRACK]` rows, evaluates it per source (joint / slot / per-slot accuracy, slot F1, invalid
JSON), and pushes it to `makan-where-cs425/makan-qwen-track`, which the backend serves as
`MAKAN_TRACKER_ADAPTER`. All training code is in the script; this notebook only sets up and calls it.

- **Colab:** GPU runtime (T4 is enough), secrets `HF_TOKEN` (write access, to push) and `GITHUB_TOKEN`
  (read access, to clone the repo), each with notebook access. Checkpoints go to Drive, so after a
  disconnect, run all again and training resumes from the newest one.
- **GPU cluster / any Jupyter:** start Jupyter from `training/` in a checkout, with
  `backend/requirements-ml.txt` + `bitsandbytes` installed and `HF_TOKEN` exported.

In [ ]:
QUICK_TRIAL = False   # True: 200 rows, 1 epoch, 20 eval rows - check the setup in minutes before the real run
EXTRA_ARGS = "--no-4bit --limit 4000"       # Use plain 16-bit LoRA (faster throughput) and limit dataset size to 4000 rows
HUB_REPO = "makan-where-cs425/makan-qwen-track"
REPO_URL = "github.com/deniseLie/CS425-Makan-Where.git"

In [ ]:
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive, userdata
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    drive.mount("/content/drive")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    REPO = Path("/content/CS425-Makan-Where")
    OUT = Path("/content/drive/MyDrive/makan-qwen/track" + ("_trial" if QUICK_TRIAL else ""))
    if not REPO.exists():
        url = f"https://{userdata.get('GITHUB_TOKEN')}@{REPO_URL}"
        # no check=True: its error message would print the URL, token included
        if subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(REPO)]).returncode:
            raise RuntimeError("git clone failed: check the GITHUB_TOKEN secret")
        subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", f"https://{REPO_URL}"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft", "accelerate", "bitsandbytes", "torchao"], check=True)
else:
    REPO = Path.cwd().parent              # run from training/ in a local checkout
    OUT = REPO / "outputs" / ("qwen_track_trial" if QUICK_TRIAL else "qwen_track")   # outputs/ is gitignored

SCRIPT = REPO / "training" / "qwen_track.py"
FINAL = OUT / "final"
assert SCRIPT.exists(), f"{SCRIPT} missing: is qwen_track.py pushed to GitHub main?"
ARGS = f"--out '{OUT}' " + ("--limit 200 --epochs 1 --eval-n 20 " if QUICK_TRIAL else "") + EXTRA_ARGS
print("script:", SCRIPT, "| out:", OUT, "| args:", ARGS)

Mounted at /content/drive
script: /content/CS425-Makan-Where/training/qwen_track.py | out: /content/drive/MyDrive/makan-qwen/track | args: --out '/content/drive/MyDrive/makan-qwen/track' --no-4bit --limit 4000


In [ ]:
# Clean up old incompatible checkpoints to prevent resume errors
import shutil
from pathlib import Path

checkpoint_dir = Path("/content/drive/MyDrive/makan-qwen/track")
if checkpoint_dir.exists():
    print(f"Cleaning old checkpoints in {checkpoint_dir}...")
    for item in checkpoint_dir.iterdir():
        if item.is_dir() and item.name.startswith("checkpoint-"):
            print(f"Removing {item.name}")
            shutil.rmtree(item)
    print("Done! You can now run the training cell below safely.")
else:
    print("No existing checkpoint directory found. You are ready to train!")

Cleaning old checkpoints in /content/drive/MyDrive/makan-qwen/track...
Removing checkpoint-450
Removing checkpoint-500
Done! You can now run the training cell below safely.


## Train and evaluate on dev

Prints the training log (dev loss about every 10% of the run), then the dev scores for MultiWOZ and
synthetic separately. Predictions go to `OUT/eval_track_dev_<source>.jsonl`; the adapter to `OUT/final`.

In [ ]:
!python "{SCRIPT}" {ARGS}

train: Counter({'multiwoz': 2341, 'synthetic': 1659}) | dev loss rows: 400
config.json: 100% 660/660 [00:00<00:00, 2.54MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 3.40MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 77.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 129MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 131MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   4% 109M/3.09G [00:02<00:31, 94.7MB/s, 8.33MB/s  ] 
model.safetensors: reconstructing file:   2% 70.5M/3.09G [00:02<01:21, 37.2MB/s, 4.13kB/s  ]
model.safetensors: downloading bytes:   8% 253M/3.09G [00:03<00:16, 175MB/s, 20.5MB/s  ]
model.safetensors: reconstructing file:   3% 88.8M/3.09G [00:03<01:42, 29.4MB/s, 7.34MB/s  ]
model.safetensors: downloading bytes:   9% 280M/3.09G [00:03<00:17, 158MB/s, 23.7MB/s  ]
model.safetensors: downloading bytes:  10% 302M/3.09G [00:0

In [ ]:
# Check the active GPU and its current memory usage
!nvidia-smi

Wed Nov  4 02:38:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Test scores (for the report)

The full MultiWOZ test set (the synthetic data has no test split). Run this once, on the final adapter.

In [ ]:
!python "{SCRIPT}" --eval-only "{FINAL}" --split test --eval-n 100 --out "{OUT}"

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:10<00:00, 31.08it/s]
[TRACK] test / multiwoz: {'n': 100, 'joint_acc': 0.79, 'slot_acc': 0.955, 'slot_f1': 0.93, 'feedback_acc': 1.0, 'invalid_json': 0.0, 'acc_cuisine': 0.91, 'acc_area': 0.9, 'acc_budget': 0.93, 'acc_dietary': 1.0, 'acc_setting': 1.0, 'acc_time': 0.99}


## Push to the Hub

Only once the dev scores look good. Uploads `OUT/final` (adapter + tokenizer) to the private repo; the
Hub keeps earlier versions in its history. Then serve it with `MAKAN_TRACKER=hf` in the backend's `.env`.

In [ ]:
from huggingface_hub import HfApi

assert not QUICK_TRIAL, "that was a trial run: set QUICK_TRIAL = False and train for real first"
api = HfApi()
api.create_repo(HUB_REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=str(FINAL), repo_id=HUB_REPO, commit_message="[TRACK] adapter")
print(f"pushed https://huggingface.co/{HUB_REPO}")

pushed https://huggingface.co/makan-where-cs425/makan-qwen-track
